# Activation-Distribution-Aware LoRA Merging: Stage E Authentic GPU Training

**Project:** Activation-Distribution-Aware LoRA Merging  
**Target Base Model:** `Qwen/Qwen2.5-0.5B-Instruct`  
**Benchmark Tasks:** BoolQ (`google/boolq`), GSM8K (`openai/gsm8k`, `main`), ARC-Easy (`allenai/ai2_arc`, `ARC-Easy`)  
**Verified Baseline Commit:** `a11929c8b14c38eec7cdf2b5014b99a22e13cff5`  

---

### Research Protocol & Integrity Standards
1. **Authentic Datasets:** Real Hugging Face benchmarks only. Synthetic fallback is strictly forbidden in final mode.
2. **Benchmark Split Preservation:** Official evaluation partitions (BoolQ: 3,270, GSM8K: 1,319, ARC-Easy: 2,376) are 100% held-out and untouched.
3. **Calibration & Validation Isolation:** 128 calibration examples per task are reserved exclusively from training sources and never used for gradient updates.
4. **Exact Hyperparameter Parity:** $r=8, \alpha=16$, dropout=0.05, `q_proj`, `v_proj`, lr=$2\times 10^{-4}$, 3 epochs, batch size 4, grad accum 4.

## 1. Hardware & Environment Diagnostics (CUDA GPU Check)

Colab must be configured with a GPU runtime (Runtime > Change runtime type > T4, V100, or A100 GPU).
This cell checks GPU availability, VRAM, RAM, disk space, and aborts immediately if CUDA is absent.

In [ ]:
import sys
import shutil
import platform
import torch

print("=" * 70)
print(" HARDWARE & RUNTIME DIAGNOSTICS ")
print("=" * 70)

print(f"Python Version       : {sys.version.split()[0]}")
print(f"PyTorch Version      : {torch.__version__}")
print(f"CUDA Available       : {torch.cuda.is_available()}")

# Check CUDA hardware
if not torch.cuda.is_available():
    raise SystemError(
        "CRITICAL ERROR: No CUDA GPU detected in this Colab session!\n"
        "Please go to: Runtime -> Change runtime type -> Hardware accelerator -> Select T4, V100, or A100 GPU."
    )

gpu_name = torch.cuda.get_device_name(0)
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"GPU Model            : {gpu_name}")
print(f"GPU Total VRAM       : {total_vram_gb:.2f} GB")
print(f"CUDA Capability      : {torch.cuda.get_device_capability(0)}")

# Check RAM and Disk
try:
    import psutil
    ram = psutil.virtual_memory()
    print(f"System RAM           : {ram.available / (1024 ** 3):.2f} GB available / {ram.total / (1024 ** 3):.2f} GB total")
except ImportError:
    pass

total_d, used_d, free_d = shutil.disk_usage(".")
print(f"Disk Space           : {free_d / (1024 ** 3):.2f} GB free / {total_d / (1024 ** 3):.2f} GB total")
print("=" * 70)
print("Hardware Check: PASSED. CUDA GPU is active and ready for training.")

## 2. Mount Google Drive for Persistent Artifact Storage

Colab local disk `/content` is ephemeral and erased when sessions disconnect.
Mounting Google Drive ensures trained adapter checkpoints (`safetensors`), logs, and manifests persist permanently.

In [ ]:
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    persistent_base = Path('/content/drive/MyDrive/activation_aware_lora_merging_artifacts')
    persistent_base.mkdir(parents=True, exist_ok=True)
    print(f"Google Drive mounted successfully. Persistent storage: {persistent_base}")
except Exception as e:
    print(f"Note: Running outside standard Colab drive mount ({e}). Artifacts will be saved locally.")
    persistent_base = Path("artifacts")
    persistent_base.mkdir(parents=True, exist_ok=True)

## 3. Install Repository Dependencies (Without Breaking CUDA)

We do NOT reinstall PyTorch since Colab comes with a preconfigured CUDA-enabled PyTorch build.
We install only the necessary packages conforming to the repository's `requirements.txt`.

In [ ]:
# Install compatible ecosystem dependencies
!pip install -q transformers peft datasets accelerate safetensors pyyaml pytest scipy psutil

## 4. Repository Verification & Pinned Commit Alignment

Ensure we are in the repository directory and verify the Git commit hash matches the certified Stage E baseline.

In [ ]:
import os
import subprocess

# If running inside Colab and repo is not yet cloned:
if not os.path.exists("configs/training.yaml"):
    !git clone https://github.com/your-username/activation-aware-lora-merging.git
    %cd activation-aware-lora-merging

# Verify current git commit
commit_hash = subprocess.check_output(["git", "rev-parse", "HEAD"]).decode().strip()
print(f"Current Git Commit SHA: {commit_hash}")

# Display recent git log
!git log -1 --oneline

## 5. Authentic Dataset Provenance & Benchmark Split Audit

Execute `scripts/audit_datasets.py` with `configs/experiment.yaml`.
This confirms:
1. Pinned Hugging Face revisions loaded:
   - `google/boolq`: `35b264d03638db9f4ce671b711558bf7ff0f80d5`
   - `openai/gsm8k`: `740312add88f781978c0658806c59bc2815b9866`
   - `allenai/ai2_arc`: `210d026faf9955653af8916fad021475a3f00453`
2. Official evaluation partitions (3,270 / 1,319 / 2,376) 100% held out.
3. Exactly 2 upstream duplicate questions in ARC-Easy excluded from training candidates.
4. Zero data leakage across splits.

In [ ]:
!python scripts/audit_datasets.py --config configs/experiment.yaml

## 6. Preflight Training Check

Run `scripts/train_all_adapters.py --preflight` to verify model architecture and hyperparameter parity.

In [ ]:
!python scripts/train_all_adapters.py --preflight

## 7. Run Unit Test Suite (50/50 Tests)

Confirm that all matrix utilities, SVD invariants, KnOTS algorithms, and benchmark policies pass.

In [ ]:
!pytest tests -v

## 8. Execute Authentic Multi-Adapter GPU Training (Stage E)

Fine-tune the three independent LoRA adapters on `Qwen/Qwen2.5-0.5B-Instruct` sequentially using CUDA:
- Task 1: BoolQ (Yes/No Question Answering)
- Task 2: GSM8K (Grade School Math Reasoning)
- Task 3: ARC-Easy (Science Multiple-Choice QA)

Hyperparameters strictly preserved:
- $r=8, \alpha=16$, dropout=0.05, target modules: `q_proj`, `v_proj`
- Learning rate: $2 \times 10^{-4}$ (AdamW, weight decay 0.01, linear warmup 5%)
- Epochs: 3 | Batch size: 4 | Gradient accumulation: 4 (Effective batch size: 16)

In [ ]:
!python scripts/train_all_adapters.py \
    --config configs/training.yaml \
    --experiment-config configs/experiment.yaml \
    --device cuda

## 9. Comprehensive Post-Training Adapter Verification

Audit all trained adapter checkpoints:
1. Verifies `adapter_model.safetensors`, `adapter_config.json`, `training_metadata.json`, and `loss_curve.json`.
2. Confirms weights are non-zero, finite, non-NaN, with correct LoRA dimensions ($8 \times 896$ and $896 \times 8$).
3. Confirms cross-task weight divergence (adapters have learned distinct task features).
4. Reloads adapters through PEFT and executes real forward passes on GPU.

In [ ]:
!python scripts/verify_adapters.py --device cuda

## 10. Backup Checkpoints & Audit Metadata to Google Drive

Mirror the saved adapters and audit records to Google Drive so they are permanently preserved.

In [ ]:
import shutil

source_adapters = Path("artifacts/adapters")
dest_adapters = persistent_base / "adapters"
dest_adapters.mkdir(parents=True, exist_ok=True)

print(f"Backing up trained adapters to: {dest_adapters}...")
for task in ["boolq", "gsm8k", "arc_easy"]:
    s_dir = source_adapters / task
    d_dir = dest_adapters / task
    if s_dir.exists():
        if d_dir.exists():
            shutil.rmtree(d_dir)
        shutil.copytree(s_dir, d_dir)
        print(f"  - Successfully backed up {task} adapter to {d_dir}")
    else:
        print(f"  - [WARNING] Source directory {s_dir} does not exist!")

# Backup provenance manifests
prov_src = Path("artifacts/provenance")
prov_dest = persistent_base / "provenance"
if prov_src.exists():
    if prov_dest.exists():
        shutil.rmtree(prov_dest)
    shutil.copytree(prov_src, prov_dest)
    print(f"Successfully backed up provenance manifests to {prov_dest}")

# Backup experiment verification reports
exp_src = Path("artifacts/experiment")
exp_dest = persistent_base / "experiment"
if exp_src.exists():
    if exp_dest.exists():
        shutil.rmtree(exp_dest)
    shutil.copytree(exp_src, exp_dest)
    print(f"Successfully backed up experiment verification reports to {exp_dest}")

print("=" * 70)
print("STAGE E TRAINING AND BACKUP COMPLETED.")
print("All authentic adapter checkpoints are permanently stored in Google Drive.")
print("=" * 70)